# **Introduction to FlyWire — Part 1: Neurons**

Welcome! In this hands-on session we'll explore the **FlyWire** connectome straight from Python. FlyWire is a reconstruction of *every* neuron in the brain of one adult female fruit fly (*Drosophila melanogaster*):

* The raw data is the **FAFB** volume (*Female Adult Fly Brain*, Zheng et al., 2018): ~7,000 ultra-thin brain sections imaged by electron microscopy (EM) at 4 × 4 × 40 nm per voxel.
* An automated pipeline split this image volume into ~140,000 neurons, which were then **proofread** by a community of scientists and citizen scientists (Dorkenwald et al., 2024).
* Synapses were detected automatically (Buhmann et al., 2021), and every neuron was given hierarchical **annotations** such as its cell type (Schlegel et al., 2024).

We'll use two Python packages:
1. [`fafbseg`](https://fafbseg-py.readthedocs.io): talks to the FlyWire servers: neuron IDs, meshes, skeletons, annotations, synapses
2. [`navis`](https://navis-org.github.io/navis/): **N**euron **A**nalysis and **Vis**ualization: plotting and analyzing neuron morphology

This notebook closely follows the official [FlyWire tutorials](https://fafbseg-py.readthedocs.io/en/latest/source/tutorials/flywire_intro.html).

Rather than picking a random neuron, we'll follow one famous cell through both notebooks: **the giant fiber** (GF, cell type *DNp01*). There is one GF per brain hemisphere. Each sends one of the thickest axons in the fly nervous system down the neck to the ventral nerve cord (the fly's "spinal cord"). When a looming object (say, your hand or a predator) approaches, the GF fires and triggers a fast escape take-off.

## 0. Setup

To run this notebook on Colab, [click here](https://colab.research.google.com/github/ryanrahy/fcnc_workshop/blob/master/day1/FW_intro_part1_neurons.ipynb).

### 0.1 Installation
If you are on a fresh machine or Google Colab, run the next cell (it takes a minute or two). Otherwise skip it.

In [ ]:
# get code + data
!git clone https://github.com/ryanrahy/fcnc_workshop.git
import os
os.chdir('./fcnc_workshop/day1/')

# install missing packages
!pip install navis flybrains ipywidgets plotly pyarrow
!pip install fafbseg   # only needed for online mode (see below)

### 0.2 Online or offline?
There are two ways to get the data:

* **Offline mode (default for this workshop):** we pre-downloaded everything these notebooks need into the `fw_data/` folder. `fw_offline.py` is a small stand-in for `fafbseg.flywire` that answers the *same function calls* from those files. No token is needed. The catch: you can only ask about neurons in the bundle (the explorers list them).
* **Online mode:** query the FlyWire servers live with `fafbseg`. Any neuron works, but you need a personal FlyWire token (see 0.3).

Every line of code below is the same in both modes, so you can switch to online mode later and keep using everything you learn today.

In [ ]:
OFFLINE = True   # set to False to query FlyWire live (needs fafbseg + a token)

if OFFLINE:
    import fw_offline as flywire   # needs fw_offline.py and fw_data/ next to this notebook
else:
    from fafbseg import flywire

### 0.3 Your FlyWire access token (online mode only, skip in offline mode)
FlyWire data is free, but querying it live needs a personal **API token**:

1. Log in at [codex.flywire.ai](https://codex.flywire.ai) with a Google account and accept the terms of service.
2. Go to <https://global.daf-apis.com/auth/api/v1/create_token> and copy the token it shows you (a long string of letters and numbers).
3. Paste it into the cell below, remove the `#` and run the cell **once**. The token is saved on your computer, so you won't need to do this again.

**Warning:** Treat the token like a password: don't share it or commit it to GitHub. Delete it from the cell after running.

In [ ]:
# flywire.set_chunkedgraph_secret("PASTE_YOUR_TOKEN_HERE")

### 0.4 Imports and dataset
FlyWire exists as several *datasets*. `"production"` is the live version that proofreaders keep editing (restricted access), and `"public"` contains the released data. We use the public one.

In [ ]:
# Import navis (flywire was imported above)
import navis
navis.set_loggers("ERROR")  # hide navis' chatty warnings (e.g. about soma detection)
import flybrains  # template brain meshes (e.g. the outline of the FlyWire brain)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# We will use the public dataset for this tutorial
flywire.set_default_dataset("public")

# Check which materializations are available
flywire.get_materialization_versions()

Each row is a **materialization**: a frozen snapshot of the whole dataset (neurons + synapses + annotations) taken at a given `time_stamp`. Public releases are numbered; **783** is the version used in the 2024 FlyWire papers, and it's the one we'll use. Because snapshots are frozen, everyone analyzing v783 gets exactly the same answers.

---
## 1. Root IDs: a neuron's (temporary) name tag

The automated segmentation first cuts the brain into millions of tiny fragments called **supervoxels**. A neuron is a set of supervoxels glued together, and the glued object gets a **root ID**: an 18-digit number such as `720575940622838154`.

Almost every FlyWire function takes root IDs as input, so the first task is always: *how do I get the root ID of the neuron I care about?* There are three common ways:

1. **By location:** you have an x/y/z coordinate, e.g. from clicking on a neuron in the FlyWire web viewer ([Neuroglancer](https://ngl.flywire.ai)).
2. **By name:** you search the annotations for a cell type (Section 3).
3. **By connectivity:** you look up the partners of a neuron you already have (Part 2).

### 1.1 From a location to a root ID
Suppose you were browsing the EM images and found a very thick axon at this voxel coordinate. Which neuron is it?

In [ ]:
# A point (x, y, z) in *voxel* coordinates inside a neuron of interest.
# Neuroglancer shows exactly these numbers at the top of the screen.
loc = [110241, 59613, 3393]

# Fetch the root ID at that location
flywire.locs_to_segments([loc])

We can look up several locations at once. The second point below is in the *other* brain hemisphere:

In [ ]:
locs = [[110241, 59613, 3393],
        [158752, 46432, 4776]]
# One root ID per location, in the same order as `locs`
flywire.locs_to_segments(locs)

These two IDs are the left and right giant fibers. We'll keep them in variables so we don't have to type 18 digits again:

In [ ]:
GF_LEFT = 720575940622838154
GF_RIGHT = 720575940632499757

**A note on units.** Voxels in FAFB are not cubes: each one is 4 nm × 4 nm in x/y but 40 nm in z (the section thickness). Locations in Neuroglancer and the annotation tables are in *voxels*. Meshes, skeletons and synapse positions are in *nanometres*. Converting is just a multiplication:

In [ ]:
voxel_size = np.array([4, 4, 40])   # nm per voxel in x, y, z
# Multiply each coordinate by its voxel size to convert voxels to nm
loc_nm = np.array(loc) * voxel_size
loc_nm

### 1.2 Root IDs change over time
Root IDs are **not permanent**. Every proofreading edit (merging two pieces, or splitting off a wrongly attached piece) creates a *new* root ID for the edited neuron. Supervoxels, however, never change. So the *same* location can belong to different root IDs at different points in time.

Let's look up our point at two public releases: v630 (spring 2023) and v783 (autumn 2023).

In [ ]:
# The same point at two releases; [0] takes the single root ID out of the returned array
root_630 = flywire.locs_to_segments([loc], timestamp="mat_630")[0]
root_783 = flywire.locs_to_segments([loc], timestamp="mat_783")[0]

print("root ID at v630:", root_630)
print("root ID at v783:", root_783)

If the two numbers differ, the giant fiber was edited between the two releases. `is_latest_root` tells us whether an ID is still current, and `update_ids` maps old IDs to their newest version:

In [ ]:
# Check if root IDs are outdated (i.e. have more recent edits)
flywire.is_latest_root([root_630, root_783])

In [ ]:
# Map (possibly outdated) IDs to the most recent version
flywire.update_ids([root_630, root_783])

**Always** note which materialization your root IDs come from. A list of IDs from a 2023 paper may not work against today's live data, but it will always work against the materialization the paper used.

---
## 2. Looking at neurons

FlyWire neurons come in two flavors:

* **Meshes:** the 3D surface of the neuron, exactly as segmented. Accurate, but heavy (hundreds of thousands to millions of triangles).
* **Skeletons:** a stick-figure tree of nodes and edges running through the middle of the neuron. Much lighter, and ideal for measuring things like cable length, branching, or distance along the neuron.

### 2.1 Meshes

In [ ]:
# The giant fiber is one of the biggest neurons in the brain. To keep the download
# small we fetch a lower-resolution mesh (lod=3, "level of detail") from the
# static v783 segmentation. For an average neuron the default call
# flywire.get_mesh_neuron(root_id) works fine.
n = flywire.get_mesh_neuron(GF_LEFT, dataset="flat_783", lod=3)
n

Even at low resolution this mesh has close to a million triangles (`faces`)! `navis.plot2d()` draws neurons with matplotlib. The `view` argument picks the viewing angle: `("x", "-y")` is the classic **frontal view**, as if you were looking the fly in the eyes. We also add the outline of the FlyWire brain from `flybrains` for orientation.

In [ ]:
fig, ax = navis.plot2d([n, flybrains.FLYWIRE], color=["#eb6834", (0.8, 0.8, 0.8, 0.1)], view=("x", "-y"))
ax.set_title("Left giant fiber (mesh), frontal view");

Notice the bushy dendrites on the side of the brain, where the GF collects visual input, and the thick axon that loops toward the midline and leaves the brain at the bottom through the neck.

### 2.2 Skeletons
The public release comes with high-quality, precomputed skeletons for all proofread neurons (neurons without one can be skeletonized with `flywire.skeletonize_neuron()`).

In [ ]:
sk = flywire.get_skeletons(GF_LEFT)
sk

The summary already includes some morphology. Units are nanometres, so let's convert and compare the two representations:

In [ ]:
# `:>9,` right-aligns a number and adds thousands separators; cable length is in nm, so / 1000 gives µm
print(f"Mesh:     {n.n_vertices:>9,} vertices, {n.n_faces:,} faces")
print(f"Skeleton: {sk.n_nodes:>9,} nodes, {sk.n_branches:,} branches, {sk.n_leafs:,} tips")
print(f"Total cable length: {sk.cable_length / 1000:,.0f} µm")

**A small detour: how many somas does a neuron have?** Look at the `soma` entry in the skeleton summary above. Instead of one node, it lists hundreds! The automatic soma detection flags any node with a large radius as "soma," and the giant fiber's axon is so thick that it fools the detector. Real data is messy, and it's worth sanity-checking automated labels:

In [ ]:
# sk.soma is a single node ID or an array of them; np.atleast_1d makes len() work for both
print("Nodes labeled as soma:", len(np.atleast_1d(sk.soma)))

# For this neuron the skeleton's root node sits right on the cell body, so we use that instead
sk.soma = sk.root[0]
print("Soma node after fix:  ", sk.soma)

Now let's overlay the skeleton on the (transparent) mesh to see how well it captures the shape:

In [ ]:
fig, ax = navis.plot2d([sk, n], color=["k", (0.92, 0.41, 0.2, 0.15)], view=("x", "-y"))
ax.set_title("Skeleton (black) inside the mesh");

### 2.3 Interactive 3D
`navis.plot3d()` creates an interactive 3D view in the notebook: **drag** to rotate, **scroll** to zoom, **double-click** to reset, and click legend entries to hide or show objects. Let's show both giant fibers in the brain.

(Skeletons work much better than million-triangle meshes in the browser.)

In [ ]:
gfs = flywire.get_skeletons([GF_LEFT, GF_RIGHT])

navis.plot3d([gfs, flybrains.FLYWIRE], color=["#2a78d6", "#eb6834"], backend="plotly")

You can also open neurons in the FlyWire web viewer, which shows the actual EM images. `encode_url` builds a Neuroglancer link with our neurons selected. (Viewing it requires logging in with a Google account in your browser, but no token. In offline mode you get a pre-made link.)

In [ ]:
# Build a Neuroglancer link with both giant fibers selected
url = flywire.encode_url(segments=[GF_LEFT, GF_RIGHT])
print(url)


---
## 3. Annotations: what *is* this neuron?

So far we have a root ID and a shape. To learn what the neuron *is*, we use annotations. There are two main types we can access programmatically:

* **Community annotations:** free-form labels added by FlyWire users (*"giant fiber"*, *"putative DNp01"*, …). Useful but messy.
* **Hierarchical annotations:** a systematic, whole-brain classification from Schlegel et al. (2024) that every neuron has. The levels go from coarse to fine:

| field | meaning | example values |
|---|---|---|
| `flow` | direction of information relative to the brain | afferent (in), efferent (out), intrinsic |
| `super_class` | broad class | sensory, optic, central, descending, motor, … |
| `cell_class`, `cell_sub_class` | finer groupings | e.g. ALPN, Kenyon_Cell, … |
| `cell_type` | the actual cell type | e.g. DNp01, LC4, MBON01 |
| `hemibrain_type` | matching type in the *hemibrain* connectome (Janelia) | e.g. Giant Fiber |
| `top_nt`, `top_nt_conf` | predicted neurotransmitter + confidence | acetylcholine, GABA, glutamate, … |
| `side` | hemisphere containing the soma | left / right |

### 3.1 Community annotations

In [ ]:
comm_ann = flywire.search_community_annotations(GF_LEFT)
comm_ann.head()

### 3.2 Hierarchical annotations
Now the hierarchical annotations for the same neuron. We transpose (`.T`) the one-row table so it's easier to read:

In [ ]:
hch_ann = flywire.search_annotations(GF_LEFT)
hch_ann.T

Most fields confirm what we saw. The GF is **efferent** (its output leaves the brain), it is a **descending** neuron, and it has the cell type **DNp01**, which matches the hemibrain's "Giant Fiber."

Now look at `top_nt` and `top_nt_conf` for the two GFs:

In [ ]:
# Annotations of both giant fibers, keeping only a few columns
flywire.search_annotations([GF_LEFT, GF_RIGHT])[["root_id", "side", "cell_type", "top_nt", "top_nt_conf"]]

### 3.3 Searching for neurons by name
Search terms are matched against **all** annotation fields:

In [ ]:
flywire.search_annotations("DNp01")

Free-text search finds the term *anywhere* in *any* annotation field (as a case-insensitive regular expression). So `"LC4"` also matches `LC40`, `LC41`, `LPLC4`, … and neurons where "lc4" is part of some other field:

In [ ]:
# Count the cell types among all matches and show the 10 most common
flywire.search_annotations("LC4").cell_type.value_counts().head(10)

For precise searches, use the `NeuronCriteria` object. It matches exactly by default, it can combine several fields, and it can be passed to many functions that take root IDs.

**LC4** and **LPLC2** are two types of visual projection neurons. They respond to looming objects and, as we'll see in Part 2, are major inputs to the giant fiber. Let's get them:

In [ ]:
# Short alias, so we can write NC(...) instead of flywire.NeuronCriteria(...)
NC = flywire.NeuronCriteria

# Neurons whose cell type (or hemibrain type) is exactly LC4, on the left side
lc4_left = flywire.search_annotations(NC(type="LC4", side="left"))
print(len(lc4_left), "LC4 neurons on the left")
lc4_left.head()

(Note: the `type` argument is a special argument that searches against all `*_type` fields in the annotations, i.e. currently against both `cell_type` and `hemibrain_type`.)

We can pass the same criteria straight to `get_skeletons` to fetch and plot all of them at once:

In [ ]:
# NeuronCriteria also works directly as input, e.g. to fetch skeletons
lc4_sk = flywire.get_skeletons(NC(type="LC4", side="left"))

fig, ax = navis.plot2d(flybrains.FLYWIRE, color=(0.85, 0.85, 0.85, 0.1), view=("x", "-y"))
navis.plot2d(lc4_sk, color="#2a78d6", lw=0.3, ax=ax, view=("x", "-y"))
navis.plot2d(sk, color="#eb6834", lw=0.6, ax=ax, view=("x", "-y"))
ax.set_title(f"{len(lc4_sk)} LC4 neurons (blue) and the left giant fiber (orange)");

Each LC4 neuron samples a small patch of the eye in the optic lobe (far left). All their axons then converge into one tight bundle, an optic glomerulus, right where the GF dendrites are. Anatomy already hints at the connectivity we'll measure in Part 2!

### 3.4 A census of the whole brain
`search_annotations(None)` returns the annotation table for all ~140k neurons. With pandas we can summarize the whole brain in one line:

In [ ]:
# None = no search term: the annotation table of all neurons
all_ann = flywire.search_annotations(None)
# Number of neurons per super class, from smallest to largest
counts = all_ann.super_class.value_counts().sort_values()

counts

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(counts.index, counts.values, color="#2a78d6", height=0.7)
for y, v in enumerate(counts.values):
    ax.text(v + 500, y, f"{v:,}", va="center", fontsize=8)
ax.set_xlabel("number of neurons")
ax.set_title("FlyWire neurons per super class")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

Over half the neurons in the fly brain belong to the visual system (`optic` + `visual_projection`)!

---
## 4. Small exercises

### Exercise 1
A colleague sends you a coordinate: **`[107560, 34947, 4823]`** (voxels). They say it's "a really weird neuron." Find out:

1. its root ID,
2. its cell type, super class and predicted neurotransmitter,
3. what it looks like (frontal view, with the brain outline).

<details><summary>Show solution</summary>

```python
mystery = flywire.locs_to_segments([[107560, 34947, 4823]])[0]
display(flywire.search_annotations(mystery)[["root_id", "super_class", "cell_class", "cell_type", "top_nt", "top_nt_conf"]])

mystery_sk = flywire.get_skeletons(mystery)
fig, ax = navis.plot2d([mystery_sk, flybrains.FLYWIRE], color=["#2a78d6", (0.8, 0.8, 0.8, 0.1)], view=("x", "-y"))
```
It is the **APL** neuron (anterior paired lateral), with just one per hemisphere. This single GABAergic neuron spreads through the entire mushroom body (the fly's learning center) and provides feedback inhibition to the ~2,600 Kenyon cells there.
</details>

In [ ]:
# your code here

### Exercise 2

Find a cell type with only one neuron per hemisphere and one with more than 1,000. (Hint: `all_ann.cell_type.value_counts()`.)

In [ ]:
# your code here

### Exercise 3

Plot `DA1_lPN` and `MBON01` from the `top` view. DA1 projection neurons carry smell information *into* the mushroom body calyx at the back of the brain, while MBON01 reads information *out* of the mushroom body lobes toward the front. Can you see this?

In [ ]:
# your code here

### Exercise 4

Which neurotransmitter is most common among all `central` neurons? Among `visual_projection` neurons?

<details><summary>Show solution for 3</summary>

```python
for sc in ["central", "visual_projection"]:
    print(sc, all_ann[all_ann.super_class == sc].top_nt.value_counts(normalize=True).round(2).head(3).to_dict())
```
Acetylcholine wins in both. But ~18 % of central neurons are predicted to be **dopaminergic**, which is suspicious. Dig in:

```python
central = all_ann[all_ann.super_class == "central"]
central[central.top_nt == "dopamine"].cell_class.value_counts().head()
```
Almost all of them are **Kenyon cells**, which are known to be cholinergic (see the `known_nt` column). The predictor gets this whole class wrong. It's another reminder to treat predictions as hypotheses!
</details>

In [ ]:
# your code here

---
## Summary

* **Root IDs** identify neurons, but they change with every edit. Always keep track of the **materialization** (we use v783).
* Coordinates come in **voxels** (4 × 4 × 40 nm) or **nanometres**.
* **Meshes** are accurate but heavy. **Skeletons** are light and good for measuring.
* **Annotations** tell us what a neuron is. Use `NeuronCriteria` for exact searches, and treat predictions (like neurotransmitters) with healthy skepticism.

In part 2, we look at the giant fiber's **synapses** and find out which neurons talk to it, and how strongly.